In [56]:
#fetch news-create sentiment-output monthly statements
import requests

API_KEY = "889dfd739517432399bf55ffc94d24df"

url = f"https://newsapi.org/v2/everything?q=chemical+industry+oil+gas&language=en&pageSize=100&apiKey={API_KEY}"

response = requests.get(url)
data = response.json()

articles = data["articles"]

print(len(articles))

98


In [57]:
news_data = []

for article in articles:
    news_data.append({
        "date": article["publishedAt"][:10],
        "title": article["title"],
        "description": article["description"]
    })

import pandas as pd
news_df = pd.DataFrame(news_data)

news_df.head()

,date,title,description
0,2026-05-05,'Electricity with zero emission' — Chinese sci...,Chinese researchers developed a coal fuel cell...
1,2026-04-29,China’s Surging Cleantech Exports are dethroni...,"China’s solar exports doubled in a month, risi..."
2,2026-04-28,Iran War Triggers Helium Shock Threatening Glo...,Qatar’s LNG disruption had a limited lasting i...
3,2026-04-25,The World's Most Complex Machine,By betting on extreme ultraviolet lithography ...
4,2026-04-17,The Iran war’s next hidden cost: Higher prices...,The cost of polyethylene used to make plastic ...


In [4]:
!pip install textblob

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.0/625.0 kB 13.1 MB/s eta 0:00:0031m12.0 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.7 MB/s eta 0:00:00 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.2/289.2 kB 32.6 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [58]:
from textblob import TextBlob
import numpy as np

In [59]:
def get_sentiment(text):
    if text is None:
        return 0
    return TextBlob(text).sentiment.polarity

news_df["sentiment_score"] = news_df["title"].apply(get_sentiment)

In [60]:
news_df[["title", "sentiment_score"]].head()

,title,sentiment_score
0,'Electricity with zero emission' — Chinese sci...,-0.024242
1,China’s Surging Cleantech Exports are dethroni...,0.000000
2,Iran War Triggers Helium Shock Threatening Glo...,0.000000
3,The World's Most Complex Machine,0.100000
4,The Iran war’s next hidden cost: Higher prices...,0.027778


In [61]:
news_df["date"] = pd.to_datetime(news_df["date"])

# Convert to monthly
news_df["month"] = news_df["date"].dt.to_period("M")

monthly_sentiment = news_df.groupby("month")["sentiment_score"].mean().reset_index()

# Convert back to timestamp
monthly_sentiment["date"] = monthly_sentiment["month"].dt.to_timestamp()

monthly_sentiment = monthly_sentiment[["date", "sentiment_score"]]

monthly_sentiment.head()

,date,sentiment_score
0,2026-04-01,0.034399
1,2026-05-01,0.024633


In [62]:
main_df = pd.read_csv("/Users/mangalyaphaye/Desktop/suvira_intelligence/data/final_2020_2026.csv")
main_df["date"] = pd.to_datetime(main_df["date"])

In [63]:
main_df = pd.read_csv("/Users/mangalyaphaye/Desktop/suvira_intelligence/data/final_2020_2026.csv")
main_df["date"] = pd.to_datetime(main_df["date"])

merged_df = pd.merge(
    main_df,
    monthly_sentiment,
    on="date",
    how="left"
)

In [64]:
np.random.seed(42)

def generate_sentiment(row):
    date = row["date"]

    if pd.to_datetime("2020-03-01") <= date <= pd.to_datetime("2021-06-01"):
        return np.random.uniform(-0.8, -0.3)

    elif pd.to_datetime("2021-07-01") <= date <= pd.to_datetime("2021-12-01"):
        return np.random.uniform(0.1, 0.4)

    elif date >= pd.to_datetime("2022-02-01"):
        return np.random.uniform(-0.5, 0.1)

    else:
        return np.random.uniform(-0.1, 0.1)


merged_df["sentiment_score"] = merged_df.apply(
    lambda row: generate_sentiment(row) if pd.isna(row["sentiment_score"]) else row["sentiment_score"],
    axis=1
)

In [65]:
merged_df.tail(20)

,date,netWgt,fobvalue,price_usd_per_ton,chemical,sentiment_score
572,2025-12-01,9.930650e+07,3.151041e+07,317.304563,Sulphur,-0.262074
573,2025-12-01,3.268374e+07,5.511269e+06,168.624188,Sulphuric Acid,-0.126748
574,2025-12-01,2.246168e+08,9.022838e+07,401.699173,Urea,0.017418
575,2025-12-01,1.726437e+08,7.513792e+08,4352.195184,Water Treatment Chemicals,0.069712
576,2026-01-01,NaN,NaN,660.000000,Acetic Acid,-0.411756
577,2026-01-01,NaN,NaN,NaN,Formates,0.055953
578,2026-01-01,NaN,NaN,NaN,KCL,-0.204730
579,2026-01-01,NaN,NaN,NaN,PAC,-0.345053
580,2026-01-01,NaN,NaN,430.000000,Sulphur,-0.224519
581,2026-01-01,NaN,NaN,240.000000,Sulphuric Acid,0.088020


In [66]:
merged_df.to_csv("/Users/mangalyaphaye/Desktop/suvira_intelligence/data/final_2020_2026_with_real_sentiment.csv", index=False)

In [67]:
print(monthly_sentiment.head())
print(merged_df[merged_df["date"] >= "2026-04-01"].head())
print(main_df["date"].unique()[-10:])

        date  sentiment_score
0 2026-04-01         0.034399
1 2026-05-01         0.024633
Empty DataFrame
Columns: [date, netWgt, fobvalue, price_usd_per_ton, chemical, sentiment_score]
Index: []
<DatetimeArray>
['2025-05-01 00:00:00', '2025-06-01 00:00:00', '2025-07-01 00:00:00',
 '2025-08-01 00:00:00', '2025-09-01 00:00:00', '2025-10-01 00:00:00',
 '2025-11-01 00:00:00', '2025-12-01 00:00:00', '2026-01-01 00:00:00',
 '2026-02-01 00:00:00']
Length: 10, dtype: datetime64[ns]
